# Módulo 4: Machine Learning en la Industria
## Día 08 - Master Class: Mega-Pipeline y Preparación para Producción

En esta sesión, el enfoque se trasladará del análisis en entornos de desarrollo a la construcción de **Software de Producción**. Se estudiará cómo estructurar un proyecto de Inteligencia Artificial utilizando Pipelines para garantizar su escalabilidad y robustez frente a datos del mundo real.

---

### **Sección 1: Teoría - Machine Learning en la Industria (Casos de Estudio)**

Previo a la codificación, es fundamental comprender cómo se monetizan y aplican estos algoritmos en el sector industrial.

#### **1. Marketing y Retail: El Abandono (Churn) y la Cesta de Compra**
*   **Market Basket Analysis (Análisis de Cesta):** Algoritmos de asociación (como Apriori) analizan millones de tickets de supermercado para descubrir relaciones de consumo. Por ejemplo: "si un cliente compra pañales los viernes, existe un 80% de probabilidad de que también adquiera cerveza". Este tipo de análisis define la distribución logística y física en grandes supermercados.
*   **Predicción de Churn (Fuga de Clientes):** Mediante ensambles (XGBoost/Random Forest), las empresas de telecomunicaciones logran identificar con meses de anticipación qué clientes están en riesgo de cancelar un servicio. 
    *   *El Riesgo Financiero:* Si el modelo falla y predice falsamente la lealtad (Falso Negativo), la empresa no tomará acción preventiva y perderá los ingresos asociados a dicho cliente.

#### **2. Medicina y Salud: El Costo de un Falso Negativo**
*   Los modelos de aprendizaje profundo y ensambles son utilizados ampliamente para detectar anomalías tempranas en radiografías y estudios genéticos.
*   *Ética y Finanzas:* En el ámbito médico, se busca optimizar la sensibilidad (Recall). Es preferible generar una falsa alarma en pacientes sanos (Falso Positivo) que requiera estudios adicionales precautorios, a incurrir en el error fatal de enviar a casa a un paciente enfermo asumiendo erróneamente que está sano (Falso Negativo).

#### **3. Smart Cities e Industria 4.0**
*   Se emplean algoritmos analíticos sobre el tráfico vehicular en tiempo real para sincronizar dinámicamente las redes de semáforos, o se proyecta la demanda energética para que las plantas térmicas optimicen sus ciclos de generación de forma anticipada.

#### **4. Metodología CRISP-DM**
Para estructurar exitosamente proyectos analíticos a gran escala, la industria adopta la metodología **CRISP-DM** (Cross-Industry Standard Process for Data Mining), compuesta por 6 fases iterativas:
1.  **Business Understanding (Entendimiento del Negocio):** Definición del problema estratégico o financiero.
2.  **Data Understanding (Entendimiento de Datos):** Auditoría e inspección de la información disponible.
3.  **Data Preparation (Preparación de Datos):** Tratamiento de valores nulos, estandarización y transformación (Construcción del Pipeline).
4.  **Modeling (Modelado):** Entrenamiento iterativo de los algoritmos de Machine Learning.
5.  **Evaluation (Evaluación):** Análisis del desempeño técnico a través de matrices de confusión y traducción al impacto económico.
6.  **Deployment (Despliegue):** Integración técnica del modelo resultante en plataformas o sistemas de producción (`joblib`).

---
### **Sección 2: Entendimiento y Extracción de Datos (El inicio del Mega-Pipeline)**
En esta etapa, se analizará una base de datos proveniente del sector de Telecomunicaciones (Telco Customer Churn) con el objetivo de modelar la retención de clientes.

In [ ]:
# Importación del Ecosistema Profesional de Data Science
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento Profesional y Pipelines
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Algoritmo Robusto de la Industria
from sklearn.ensemble import RandomForestClassifier

# Herramientas de Evaluación Financiera/Matemática
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Guardado para Despliegue
import joblib

In [ ]:
# 1. Extracción de Datos (Carga del Data Lake / CSV)
# Se utiliza un repositorio público para la carga directa del set de datos.
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

# La variable 'TotalCharges' ocasionalmente se presenta como texto vacío. Se procede a corregir el tipo de dato.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

display(df.head())

In [ ]:
# 2. EDA: Análisis Exploratorio de Datos
print("--- INFORMACIÓN TÉCNICA DEL DATASET ---")
df.info()

print("\n--- RESUMEN ESTADÍSTICO ---")
display(df.describe())

print("\n--- VALORES NULOS DETECTADOS ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# 3. Visualizaciones Tácticas de Negocio
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# A. Balanceo de la Variable Objetivo (Churn)
sns.countplot(data=df, x='Churn', palette='Set2', ax=axes[0])
axes[0].set_title('Conteo de Abandono (Churn)')

# B. Histograma de Gastos Mensuales
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', kde=True, palette='bright', ax=axes[1])
axes[1].set_title('Distribución de Cargos Mensuales')

# C. Matriz de Correlación (Solo numéricas)
nums = df.select_dtypes(include=['float64', 'int64'])
sns.heatmap(nums.corr(), annot=True, cmap='coolwarm', ax=axes[2])
axes[2].set_title('Mapa de Correlación Numérica')

plt.tight_layout()
plt.show()

---
### **Sección 3: Limpieza y Feature Engineering (La Creación del Pipeline)**

#### **Pausa Didáctica: ¿Qué es el DATA LEAKAGE (Fuga de Datos)?**
El "Data Leakage" es un error crítico en Ciencia de Datos que ocurre cuando se aplican transformaciones estadísticas (como la imputación de promedios o el escalado numérico) **antes** de separar los datos en conjuntos de Entrenamiento y Prueba. Esto ocasiona que el modelo utilice inadvertidamente información del conjunto de pruebas, generando resultados que parecen perfectos en pruebas, pero deficientes en producción.

**Regla Fundamental:** Siempre se debe ejecutar el `train_test_split` previo a cualquier procesamiento iterativo. La construcción de un objeto `Pipeline` garantiza que las estadísticas de transformación matemática se calculen única y exclusivamente sobre el conjunto de entrenamiento.

In [ ]:
# 1. Definición de Variables Predictoras (X) y Objetivo (y)
# Se excluye customerID ya que es una variable sin valor analítico que introduciría ruido al modelo.
X = df.drop(columns=['Churn', 'customerID'])
y = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

# 2. División INICIAL de los datos para prevenir Data Leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Dimensiones de Datos para Entrenamiento: {X_train.shape}")
print(f"Dimensiones de Datos para Examen Final: {X_test.shape}")

#### **Arquitectura de Preprocesamiento: ColumnTransformer**
Dado que los modelos algorítmicos requieren representaciones numéricas estandarizadas, la información cruda debe ser transformada siguiendo lógicas específicas:
1.  **Variables Numéricas:** Frecuentemente presentan valores faltantes y escalas dispares. Se procederá a imputar valores utilizando la Mediana, y posteriormente se estandarizará su varianza (`StandardScaler`).
2.  **Variables Categóricas:** La imputación de nulos se realizará utilizando la clase mayoritaria (Moda). Posteriormente, se utilizará la técnica de codificación One-Hot para transformar las categorías en vectores binarios.

In [ ]:
# Identificación automática de las columnas categóricas y numéricas
cols_numericas = X_train.select_dtypes(include=['int64', 'float64']).columns
cols_categoricas = X_train.select_dtypes(include=['object']).columns

# A. PIPELINE NUMÉRICO: Operaciones aplicables exclusivamente a números
pipeline_numerico = Pipeline(steps=[
    ('imputador', SimpleImputer(strategy='median')),
    ('escalador', StandardScaler())
])

# B. PIPELINE CATEGÓRICO: Operaciones aplicables exclusivamente a variables de texto/categóricas
pipeline_categorico = Pipeline(steps=[
    ('imputador', SimpleImputer(strategy='most_frequent')),
    ('codificador', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# C. EL TRANSFORMADOR MAESTRO: Unifica ambas tuberías utilizando ColumnTransformer
preprocesador = ColumnTransformer(transformers=[
    ('num', pipeline_numerico, cols_numericas),
    ('cat', pipeline_categorico, cols_categoricas)
])

---
### **Sección 4: Construcción del Algoritmo y Optimización (GridSearchCV)**

En esta etapa, se conectará el bloque de limpieza de datos (`preprocesador`) directamente a un estimador de Machine Learning (`RandomForestClassifier`).

Adicionalmente, se integrará el método de optimización `GridSearchCV`. Esta herramienta permite realizar una búsqueda exhaustiva de hiperparámetros de manera automatizada, probando múltiples configuraciones y seleccionando matemáticamente la arquitectura óptima mediante Validación Cruzada (Cross-Validation).

In [ ]:
# 1. Ensamblaje del Pipeline unificado (Limpieza + Modelo)
mega_pipeline = Pipeline(steps=[
    ('prep', preprocesador),
    ('modelo', RandomForestClassifier(class_weight='balanced', random_state=42))
])

# 2. Definición del espacio de hiperparámetros a optimizar
# Nota técnica: Para referenciar un parámetro dentro de un paso del pipeline, se utiliza el prefijo "nombrePaso__parametro"
param_grid = {
    'modelo__n_estimators': [50, 100, 150],
    'modelo__max_depth': [5, 10, None]
}

# 3. Configuración del optimizador GridSearch
optimizador = GridSearchCV(
    estimator=mega_pipeline, 
    param_grid=param_grid, 
    cv=5,
    scoring='f1',         # Se busca un equilibrio armónico entre Precisión y Sensibilidad (Recall)
    n_jobs=-1             # Activa el uso en paralelo de todos los núcleos del procesador
)

print("Iniciando el entrenamiento concurrente de combinaciones algorítmicas...")
# 4. Ejecución del entrenamiento
optimizador.fit(X_train, y_train)

print("\n¡Entrenamiento Completado!")
print(f"Mejor configuración de hiperparámetros identificada: {optimizador.best_params_}")

---
### **Sección 5: Evaluación a Nivel Profesional y Exportación (Cierre del Proyecto)**

Una vez obtenido el modelo óptimo, este será evaluado frente al conjunto de datos de prueba (`X_test`) para determinar su verdadera capacidad predictiva y viabilidad financiera.

In [ ]:
# Extracción del modelo ganador
modelo_final = optimizador.best_estimator_

# Ejecución de predicciones sobre el conjunto de examen aislado
predicciones = modelo_final.predict(X_test)

print("--- REPORTE TÉCNICO DE CLASIFICACIÓN ---")
print(classification_report(y_test, predicciones))

In [ ]:
# Generación de la Matriz de Confusión
matriz = confusion_matrix(y_test, predicciones)
disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=['Se Queda (0)', 'Abandona (1)'])

fig, ax = plt.subplots(figsize=(6,6))
disp.plot(cmap='Blues', ax=ax)
plt.title('Matriz de Confusión - Perspectiva de Negocios')
plt.show()

#### **Análisis Financiero Directivo:**
Las métricas estadísticas subyacentes requieren una traducción objetiva al contexto de rentabilidad de la organización:

*   **Cuadrante Superior Izquierdo (Verdaderos Negativos):** Clientes estables correctamente identificados. No existen pérdidas ni costos operativos.
*   **Cuadrante Inferior Derecho (Verdaderos Positivos):** Clientes con alto riesgo de abandono detectados oportunamente, permitiendo a la organización ejecutar estrategias efectivas de retención.
*   **Cuadrante Superior Derecho (Falsos Positivos - Falsa Alarma):** Clientes estables erróneamente clasificados como riesgosos. Generan un costo operativo leve al destinarles promociones de manera innecesaria.
*   **Cuadrante Inferior Izquierdo (Falsos Negativos - Riesgo Crítico):** Clientes en fuga omitidos por el algoritmo. Representan la mayor pérdida de ingresos para la compañía, ya que no se ejecutan acciones preventivas.

**Decisión Estratégica:** Asumiendo que la deserción de un cliente tiene un impacto de $1000, y enviar una campaña promocional cuesta $20, el modelo predictivo debe priorizar una disminución exhaustiva de los *Falsos Negativos* (Maximizando el Recall).

---

### **Despliegue (Serialización para Entornos de Producción)**
El objeto Pipeline integra de manera coherente las etapas de imputación, escalado, codificación categórica y estimación. A través de la librería `joblib`, se serializará esta tubería completa en un formato binario (`.pkl`), posibilitando su integración y despliegue rápido por equipos de ingeniería de software.

In [ ]:
# Exportación profesional del artefacto para Despliegue
import joblib

joblib.dump(modelo_final, 'mega_pipeline_produccion.pkl')
print("¡Modelo serializado y guardado de manera exitosa!")
print("Artefacto generado: mega_pipeline_produccion.pkl")